In [ ]:
from lcb_runner.benchmarks import code_generation

dataset = code_generation.load_code_generation_dataset(release_version="release_v5", start_date="2024-01-01", diffulty=code_generation.Difficulty.HARD)

/Users/kaushitha/Documents/APR/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using 'test' split of the dataset.


In [ ]:
from common.llm_client import create_llm_client, LLMClient
import common.code_preprocessing as code_preprocessing

sample_problem: str = dataset[0].question_content
starter_code: str = dataset[0].starter_code 

if len(starter_code) == 0:
    starter_code = """
class Solution:
    def sol(self, input_str):
"""


llm_client: LLMClient  = create_llm_client(provider='openai', model='gpt-5')
print(f"LLM Reasoning Effort: {llm_client.reasoning_effort}")

print("Sample Problem: ")
print(sample_problem)

print("Starter Code: ")
print(starter_code)

print("=== Generating Code ===")

INITIAL_CODE_POPULATION_SIZE = 10
initial_code_prompt = f"Write {INITIAL_CODE_POPULATION_SIZE} distinct solutions to solve the following problem: {sample_problem}\n```python\n{starter_code}```. Each solution should be in a separate code block."
response = llm_client.generate(initial_code_prompt)


# formatting response
code_blocks = code_preprocessing.extract_all_code_blocks_from_response(response)

if len(code_blocks) != INITIAL_CODE_POPULATION_SIZE:
    raise ValueError(f"Expected {INITIAL_CODE_POPULATION_SIZE} code blocks, but got {len(code_blocks)}\nResponse: {response}")


for code_idx, code in enumerate(code_blocks):
    print(f"Code Block {code_idx+1}:\n{code}\n")

ImportError: cannot import name 'extract_all_code_blocks' from 'common.code_preprocessing' (/Users/kaushitha/Documents/APR/src/common/code_preprocessing/__init__.py)

In [ ]:
INITIAL_TEST_POPULATION_SIZE = 20
print("=== Generating Tests ===")
initial_test_prompt = f"Write {INITIAL_TEST_POPULATION_SIZE} simple distinct basic unit tests in a Python code block for the following problem:\n{sample_problem}\nThe solution is imported in the format:\n```python\n{starter_code}```\nWrite the tests using unittest framework. Do not use the examples givn in the problem."
print(initial_test_prompt)

response = llm_client.generate(initial_test_prompt)
print(f"response:\n{response}\n")
# formatting response
test_block = code_preprocessing.extract_code_block_from_response(response)

if not test_block:
    raise ValueError(f"Expected a test class block, but none found.\nResponse: {response}")

test_cases = code_preprocessing.analyzers.extract_test_methods_code(test_block)
if len(test_cases) != INITIAL_TEST_POPULATION_SIZE:
    raise ValueError(f"Expected {INITIAL_TEST_POPULATION_SIZE} test cases, but got {len(test_cases)}\nTest Block: {test_block}")

print(f"Test Block:\n{test_block}\n")

=== Generating Tests ===
Write 20 simple distinct basic unit tests in a Python code block for the following problem:
You are given a weighted undirected graph G with N vertices, numbered 1 to N. Initially, G has no edges.
You will perform M operations to add edges to G. The i-th operation (1 \leq i \leq M) is as follows:

- You are given a subset of vertices S_i=\lbrace A_{i,1},A_{i,2},\dots,A_{i,K_i}\rbrace consisting of K_i vertices.
For every pair u, v such that u, v \in S_i and u < v, add an edge between vertices u and v with weight C_i.

After performing all M operations, determine whether G is connected. If it is, find the total weight of the edges in a minimum spanning tree of G.

Input

The input is given from Standard Input in the following format:
N M
K_1 C_1
A_{1,1} A_{1,2} \dots A_{1,K_1}
K_2 C_2
A_{2,1} A_{2,2} \dots A_{2,K_2}
\vdots
K_M C_M
A_{M,1} A_{M,2} \dots A_{M,K_M}

Output

If G is not connected after all M operations, print -1. If G is connected, print the total w

In [ ]:
from common.sandbox import create_safe_test_environment
import numpy as np

matrix = np.zeros((INITIAL_CODE_POPULATION_SIZE, INITIAL_TEST_POPULATION_SIZE), dtype=int)
sandbox = create_safe_test_environment()

for code_idx, code in enumerate(code_blocks):
    print(f"Executing Code Block {code_idx+1}: ")
    script = code_preprocessing.builders.build_test_script_for_lcb(code, test_block)
    test_results = sandbox.execute_test_script(script)
    for test_idx, test in enumerate(test_results.test_results):
        print(f"Test: {test.name}, Status: {test.status}, Details: {test.details}")
        if test.status == "passed":
            matrix[code_idx, test_idx] = 1
        elif test.status == "failed":
            matrix[code_idx, test_idx] = 0  # failed
        else:
            matrix[code_idx, test_idx] = 0  # Let's treat errors as failures for now

Executing Code Block 1: 
Test: test_all_same_weight, Status: passed, Details: None
Test: test_big_clique, Status: passed, Details: None
Test: test_bridge_cheap, Status: passed, Details: None
Test: test_chain_pairs, Status: passed, Details: None
Test: test_disconnected_partitions, Status: passed, Details: None
Test: test_duplicate_same_operation, Status: passed, Details: None
Test: test_heavy_clique_plus_bridges, Status: passed, Details: None
Test: test_isolated_vertex, Status: passed, Details: None
Test: test_large_k_then_small, Status: passed, Details: None
Test: test_many_ops_with_redundancy, Status: failed, Details: self.assertEqual(self.S(input_str.strip()), str(expected))
AssertionError: '15' != '19'
Test: test_multiple_same_weight_edges, Status: passed, Details: None
Test: test_multiple_small_groups_connect, Status: passed, Details: None
Test: test_overlapping_choose_min, Status: passed, Details: None
Test: test_overlapping_decreasing_weights, Status: passed, Details: None
Test: 

In [ ]:
matrix

array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

In [ ]:
from common.coevolution import CoevolutionConfig, initialize_populations, update_population_beliefs
config = CoevolutionConfig(
    initial_code_population_size=INITIAL_CODE_POPULATION_SIZE,
    initial_test_population_size=INITIAL_TEST_POPULATION_SIZE,
    c0_prior=0.3,
    t0_prior=0.3, 
    alpha=0.1,
    beta=0.1,
    gamma=0.1,
)

# Updated to match the new function signature (returns only 2 arrays)
code_probs, test_probs = initialize_populations(config)
print("Initial Code Probabilities:")
print([round(a, 4) for a in code_probs.tolist()])
print("Initial Test Probabilities:")
print([round(a, 4) for a in test_probs.tolist()])

new_code_probs, new_test_probs = update_population_beliefs(code_probs, test_probs, matrix, config)


print("=== Updated Probabilities ===")
print("Code:")
print([round(a, 4) for a in new_code_probs.tolist()])

print("Test:")
print([round(a, 4) for a in new_test_probs.tolist()])


print("\n\n=== Comparison with Pass Rates ===")
print("Pass rates for each code snippet:")
# in matrix rows are code snippets and columns are tests, so we take mean across axis 1
pass_rates = matrix.mean(axis=1)
print([round(a, 4) for a in pass_rates.tolist()])

print("Pass rates for each test:")
print([round(a, 4) for a in matrix.mean(axis=0).tolist()])

Initial Code Probabilities:
[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3]
Initial Test Probabilities:
[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3]
=== Updated Probabilities ===
Code:
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Test:
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.012, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


=== Comparison with Pass Rates ===
Pass rates for each code snippet:
[0.95, 0.95, 0.95, 0.95, 0.95, 0.95, 0.95, 0.95, 0.95, 0.95]
Pass rates for each test:
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
